# 15 — Data Setelah Preprocessing dan Daftar Fitur

Notebook ini memperlihatkan **isi data yang sudah diproses**, yaitu data yang benar-benar masuk ke model skenario S1–S4, untuk ketiga modalitas:

| Bagian | Modalitas | Isi |
| :---: | :--- | :--- |
| 1 | Ringkasan | semua fitur dalam satu tabel: jumlah, bentuk, dan skenario yang memakainya |
| 2 | **Fenotipik** | tabel data per subjek + bentuknya setelah diolah untuk model |
| 3 | **rs-fMRI** | FC tangent dan FC korelasi: bentuk, nama parcel, contoh isi |
| 4 | **sMRI** | morfometri: bentuk, contoh isi per parcel |

Cara datanya dibuat dijelaskan di notebook 05 (fenotipik), 11 (rs-fMRI), dan 13 (sMRI). Notebook ini hanya **membaca** berkas yang sudah ada dan tidak mengubah apa pun. Lama proses beberapa detik.

## Persiapan

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import nibabel as nib
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "data").exists():
    ROOT = ROOT.parent
PROC = ROOT / "data" / "processed"
kohort = pd.read_csv(PROC / "kohort_dcan.csv")
F = np.load(PROC / "fitur_smri_dcan" / "fitur_smri_dcan.npz", allow_pickle=True)
FC = PROC / "fitur_dcan_22"
assert list(F["sub"]) == list(kohort["sub"])
ATLAS = {"gordon2014": 352, "hcp2016": 379, "markov2012": 205}
NAMA_HCP = [str(n) for n in nib.load(ROOT / kohort["berkas_hcp2016"].iloc[0]).header.get_axis(1).name]      # 379 parcel HCP
CONTOH = 0
print(f"{len(kohort)} subjek | subjek contoh: {kohort.loc[CONTOH, 'FILE_ID']}")

def segitiga_ke_matriks(v, n):
    M = np.zeros((n, n)); i, j = np.tril_indices(n, -1); M[i, j] = v; M[j, i] = v
    return M

1638 subjek | subjek contoh: Caltech_0051457


## 1. Ringkasan Semua Fitur

Satu subjek diwakili oleh fitur-fitur berikut. Kolom **"Dipakai di"** menunjukkan skenario mana yang memakainya, dan bagian model mana yang menerimanya (lihat diagram model di notebook 16).

In [2]:
ringkasan = pd.DataFrame([
    {"Modalitas": "Fenotipik", "Fitur": "usia, jenis kelamin, FIQ (+ penanda FIQ kosong)", "Jumlah per subjek": 4,
     "Bentuk per subjek": "4 angka", "Berkas": "data/processed/kohort_dcan.csv", "Dipakai di": "S3–S4: cabang fenotipik"},
    {"Modalitas": "rs-fMRI", "Fitur": "FC tangent Gordon (352 ROI)", "Jumlah per subjek": 352 * 351 // 2,
     "Bentuk per subjek": "segitiga bawah matriks 352 × 352", "Berkas": "fitur_dcan_22/tangent_gordon2014_gab_<fold>.npy", "Dipakai di": "S1–S4: cabang linear"},
    {"Modalitas": "rs-fMRI", "Fitur": "FC tangent HCP (379 ROI)", "Jumlah per subjek": 379 * 378 // 2,
     "Bentuk per subjek": "segitiga bawah matriks 379 × 379", "Berkas": "fitur_dcan_22/tangent_hcp2016_gab_<fold>.npy", "Dipakai di": "S1–S4: cabang linear + fitur simpul graf"},
    {"Modalitas": "rs-fMRI", "Fitur": "FC tangent Markov (205 ROI)", "Jumlah per subjek": 205 * 204 // 2,
     "Bentuk per subjek": "segitiga bawah matriks 205 × 205", "Berkas": "fitur_dcan_22/tangent_markov2012_gab_<fold>.npy", "Dipakai di": "S1–S4: cabang linear"},
    {"Modalitas": "rs-fMRI", "Fitur": "FC korelasi HCP", "Jumlah per subjek": 379 * 378 // 2,
     "Bentuk per subjek": "segitiga bawah matriks 379 × 379", "Berkas": "fitur_dcan_22/korelasi_hcp2016.npy", "Dipakai di": "S1–S4: sisi graf (20 tetangga terkuat)"},
    {"Modalitas": "sMRI", "Fitur": "morfometri: luas, volume GM, ketebalan, kelengkungan, kedalaman sulkus", "Jumlah per subjek": 360 * 5,
     "Bentuk per subjek": "360 parcel × 5", "Berkas": "fitur_smri_dcan/fitur_smri_dcan.npz (morfo_hcp2016)", "Dipakai di": "S2–S4: fitur simpul graf"},
])
display(ringkasan.style.format({"Jumlah per subjek": "{:,}"}).set_caption(f"Fitur per subjek (semua berkas berisi {len(kohort):,} subjek, urutan = kohort_dcan.csv)"))

,Modalitas,Fitur,Jumlah per subjek,Bentuk per subjek,Berkas,Dipakai di
0,Fenotipik,"usia, jenis kelamin, FIQ (+ penanda FIQ kosong)",4,4 angka,data/processed/kohort_dcan.csv,S3–S4: cabang fenotipik
1,rs-fMRI,FC tangent Gordon (352 ROI),"61,776",segitiga bawah matriks 352 × 352,fitur_dcan_22/tangent_gordon2014_gab_.npy,S1–S4: cabang linear
2,rs-fMRI,FC tangent HCP (379 ROI),"71,631",segitiga bawah matriks 379 × 379,fitur_dcan_22/tangent_hcp2016_gab_.npy,S1–S4: cabang linear + fitur simpul graf
3,rs-fMRI,FC tangent Markov (205 ROI),"20,910",segitiga bawah matriks 205 × 205,fitur_dcan_22/tangent_markov2012_gab_.npy,S1–S4: cabang linear
4,rs-fMRI,FC korelasi HCP,"71,631",segitiga bawah matriks 379 × 379,fitur_dcan_22/korelasi_hcp2016.npy,S1–S4: sisi graf (20 tetangga terkuat)
5,sMRI,"morfometri: luas, volume GM, ketebalan, kelengkungan, kedalaman sulkus","1,800",360 parcel × 5,fitur_smri_dcan/fitur_smri_dcan.npz (morfo_hcp2016),S2–S4: fitur simpul graf


## 2. Fenotipik

### 2.1 Data per subjek (sebelum diolah untuk model)

Kolom fenotipik di `kohort_dcan.csv` untuk 10 subjek pertama. `label`: 1 = ASD, 0 = Kontrol; `jenis_kelamin`: 1 = laki-laki, 2 = perempuan; FIQ kosong ditulis NaN.

In [3]:
kolom = ["FILE_ID", "dataset", "situs", "label", "usia", "jenis_kelamin", "fiq"]
display(kohort[kolom].head(10).style.set_caption("Fenotipik 10 subjek pertama"))
display(pd.DataFrame({
    "Fitur": ["usia (tahun)", "jenis kelamin", "FIQ", "label"],
    "Ringkasan": [f"{kohort['usia'].mean():.1f} ± {kohort['usia'].std():.1f} (rentang {kohort['usia'].min():.1f}–{kohort['usia'].max():.1f})",
                  f"laki-laki {(kohort['jenis_kelamin'] == 1).sum()} / perempuan {(kohort['jenis_kelamin'] == 2).sum()}",
                  f"{kohort['fiq'].mean():.1f} ± {kohort['fiq'].std():.1f}; kosong {kohort['fiq'].isna().sum()} subjek",
                  f"ASD {kohort['label'].sum()} / Kontrol {(1 - kohort['label']).sum()}"],
}).style.set_caption(f"Fenotipik seluruh kohort ({len(kohort)} subjek)"))

,FILE_ID,dataset,situs,label,usia,jenis_kelamin,fiq
0,Caltech_0051457,abide1,CALTECH,1,22.900000,1,107.000000
1,Caltech_0051458,abide1,CALTECH,1,39.200000,1,93.000000
2,Caltech_0051459,abide1,CALTECH,1,22.800000,1,106.000000
3,Caltech_0051462,abide1,CALTECH,1,20.200000,2,107.000000
4,Caltech_0051463,abide1,CALTECH,1,20.200000,2,102.000000
5,Caltech_0051465,abide1,CALTECH,1,20.200000,1,96.000000
6,Caltech_0051468,abide1,CALTECH,1,20.100000,1,100.000000
7,Caltech_0051470,abide1,CALTECH,1,29.100000,1,124.000000
8,Caltech_0051472,abide1,CALTECH,1,17.500000,1,125.000000
9,Caltech_0051473,abide1,CALTECH,1,21.200000,1,nan


,Fitur,Ringkasan
0,usia (tahun),16.1 ± 8.4 (rentang 5.2–64.0)
1,jenis kelamin,laki-laki 1299 / perempuan 339
2,FIQ,109.5 ± 15.0; kosong 129 subjek
3,label,ASD 714 / Kontrol 924


### 2.2 Setelah diolah untuk model (S3–S4)

Sebelum masuk cabang fenotipik, data diubah menjadi 4 angka. Statistiknya (rata-rata, SD, median FIQ) dihitung dari **data latih fold tersebut**. Di sini dipakai fold 0 sebagai contoh:

| Fitur model | Pengolahan |
| :--- | :--- |
| log usia | logaritma usia, lalu distandardisasi (rata-rata 0, SD 1 di data latih) |
| perempuan | 1 = perempuan, 0 = laki-laki |
| FIQ | yang kosong diisi median data latih, lalu distandardisasi |
| FIQ kosong | 1 bila FIQ aslinya kosong (supaya model tahu nilainya hasil isian) |

In [4]:
latih0 = np.where(kohort["fold_gabungan"] != 0)[0]
kosong = kohort["fiq"].isna().to_numpy()
X = np.column_stack([np.log(kohort["usia"]), (kohort["jenis_kelamin"] == 2).astype(float), kohort["fiq"], kosong.astype(float)])
X[kosong, 2] = np.nanmedian(X[latih0, 2])
for j in (0, 2):
    X[:, j] = (X[:, j] - X[latih0, j].mean()) / X[latih0, j].std()
contoh = np.r_[np.arange(5), np.where(kosong)[0][:2]]
display(pd.concat([kohort.loc[contoh, ["FILE_ID", "usia", "jenis_kelamin", "fiq"]].reset_index(drop=True),
                   pd.DataFrame(X[contoh], columns=["→ log usia (standar)", "→ perempuan", "→ FIQ (standar)", "→ FIQ kosong"]).round(3)], axis=1)
        .style.set_caption("Sebelum (kiri) → sesudah diolah (kanan); 2 baris terakhir adalah contoh subjek dengan FIQ kosong"))

,FILE_ID,usia,jenis_kelamin,fiq,→ log usia (standar),→ perempuan,→ FIQ (standar),→ FIQ kosong
0,Caltech_0051457,22.900000,1,107.000000,1.052000,0.000000,-0.192000,0.000000
1,Caltech_0051458,39.200000,1,93.000000,2.295000,0.000000,-1.152000,0.000000
2,Caltech_0051459,22.800000,1,106.000000,1.042000,0.000000,-0.261000,0.000000
3,Caltech_0051462,20.200000,2,107.000000,0.762000,1.000000,-0.192000,0.000000
4,Caltech_0051463,20.200000,2,102.000000,0.762000,1.000000,-0.535000,0.000000
5,Caltech_0051473,21.200000,1,nan,0.874000,0.000000,0.014000,1.000000
6,Leuven_2_0050722,13.800000,2,nan,-0.118000,1.000000,0.014000,1.000000


## 3. rs-fMRI

### 3.1 Bentuk data

Setiap subjek punya **matriks FC** untuk setiap atlas: angka di baris *i*, kolom *j* menyatakan hubungan sinyal fMRI parcel *i* dengan parcel *j*. Karena matriksnya simetris (*i*–*j* = *j*–*i*) dan diagonalnya tidak dipakai, yang disimpan hanya **segitiga bawahnya**. FC tangent disimpan **per fold**, karena acuannya dihitung dari data latih setiap fold (notebook 11).

In [5]:
baris = []
for a, n in ATLAS.items():
    V = np.load(FC / f"tangent_{a}_gab_0.npy", mmap_mode="r")
    baris.append({"Atlas": a, "ROI": n, "Matriks per subjek": f"{n} × {n}", "Nilai disimpan per subjek": V.shape[1],
                  "Bentuk berkas (subjek × nilai)": str(V.shape), "Berkas per atlas": "5 (satu per fold) + 1 korelasi"})
display(pd.DataFrame(baris).style.format({"Nilai disimpan per subjek": "{:,}"}).set_caption("Berkas FC tangent"))
print(f"Total fitur FC tangent 3 atlas per subjek: {sum(n * (n - 1) // 2 for n in ATLAS.values()):,}")
print(f"Contoh nama parcel HCP: {NAMA_HCP[:3]} … {NAMA_HCP[357:360]} (korteks) | {NAMA_HCP[360:363]} … (19 subkortikal)")

,Atlas,ROI,Matriks per subjek,Nilai disimpan per subjek,Bentuk berkas (subjek × nilai),Berkas per atlas
0,gordon2014,352,352 × 352,"61,776","(1638, 61776)",5 (satu per fold) + 1 korelasi
1,hcp2016,379,379 × 379,"71,631","(1638, 71631)",5 (satu per fold) + 1 korelasi
2,markov2012,205,205 × 205,"20,910","(1638, 20910)",5 (satu per fold) + 1 korelasi


Total fitur FC tangent 3 atlas per subjek: 154,317
Contoh nama parcel HCP: ['R_V1_ROI', 'R_MST_ROI', 'R_V6_ROI'] … ['L_PI_ROI', 'L_a32pr_ROI', 'L_p24_ROI'] (korteks) | ['CEREBELLUM_LEFT', 'THALAMUS_LEFT', 'CAUDATE_LEFT'] … (19 subkortikal)


### 3.2 Contoh isi: 6 parcel pertama subjek contoh (atlas HCP)

Potongan kecil dari matriks 379 × 379. **FC korelasi** (Fisher z): positif berarti sinyal searah, negatif berlawanan. **FC tangent** (fold 0): seberapa berbeda hubungan itu dibanding "otak rata-rata" subjek latih (0 = sama dengan rata-rata).

In [6]:
n6 = NAMA_HCP[:6]
kor = segitiga_ke_matriks(np.load(FC / "korelasi_hcp2016.npy", mmap_mode="r")[CONTOH], 379)[:6, :6]
tan = segitiga_ke_matriks(np.load(FC / "tangent_hcp2016_gab_0.npy", mmap_mode="r")[CONTOH], 379)[:6, :6]
display(pd.DataFrame(kor, index=n6, columns=n6).round(3).style.set_caption(f"FC korelasi (Fisher z), {kohort.loc[CONTOH, 'FILE_ID']}"))
display(pd.DataFrame(tan, index=n6, columns=n6).round(3).style.set_caption(f"FC tangent fold 0, {kohort.loc[CONTOH, 'FILE_ID']}"))

,R_V1_ROI,R_MST_ROI,R_V6_ROI,R_V2_ROI,R_V3_ROI,R_V4_ROI
R_V1_ROI,0.000000,0.525000,0.273000,1.112000,0.656000,0.059000
R_MST_ROI,0.525000,0.000000,0.053000,0.259000,0.000000,-0.328000
R_V6_ROI,0.273000,0.053000,0.000000,0.452000,0.609000,0.754000
R_V2_ROI,1.112000,0.259000,0.452000,0.000000,0.746000,0.295000
R_V3_ROI,0.656000,0.000000,0.609000,0.746000,0.000000,0.733000
R_V4_ROI,0.059000,-0.328000,0.754000,0.295000,0.733000,0.000000


,R_V1_ROI,R_MST_ROI,R_V6_ROI,R_V2_ROI,R_V3_ROI,R_V4_ROI
R_V1_ROI,0.000000,0.110000,0.033000,0.045000,0.049000,-0.069000
R_MST_ROI,0.110000,0.000000,0.114000,0.049000,0.032000,-0.046000
R_V6_ROI,0.033000,0.114000,0.000000,0.042000,0.011000,0.100000
R_V2_ROI,0.045000,0.049000,0.042000,0.000000,-0.047000,-0.045000
R_V3_ROI,0.049000,0.032000,0.011000,-0.047000,0.000000,-0.098000
R_V4_ROI,-0.069000,-0.046000,0.100000,-0.045000,-0.098000,0.000000


### 3.3 Bentuk yang masuk model

| Bagian model | Isi dari rs-fMRI |
| :--- | :--- |
| **Cabang linear** | FC tangent 3 atlas (154.317 nilai) → diproyeksikan ke komponen utama data latih (±3.900 nilai, tanpa kehilangan informasi bagi model linear; lihat notebook 16) |
| **Simpul graf** (360 parcel korteks HCP) | setiap simpul membawa **baris** matriks FC tangent parcel itu: 379 nilai (hubungannya dengan semua parcel, termasuk subkortikal) |
| **Sisi graf** (S1–S3) | setiap parcel dihubungkan ke 20 parcel dengan \|FC korelasi\| terbesar |

In [7]:
baris_tan = segitiga_ke_matriks(np.load(FC / "tangent_hcp2016_gab_0.npy", mmap_mode="r")[CONTOH], 379)[:360]
display(pd.DataFrame(baris_tan[:5, :8], index=NAMA_HCP[:5], columns=NAMA_HCP[:8]).round(3)
        .style.set_caption(f"Fitur simpul graf dari rs-fMRI: 5 simpul pertama × 8 dari 379 nilai (bentuk lengkap {baris_tan.shape})"))

,R_V1_ROI,R_MST_ROI,R_V6_ROI,R_V2_ROI,R_V3_ROI,R_V4_ROI,R_V8_ROI,R_4_ROI
R_V1_ROI,0.000000,0.110000,0.033000,0.045000,0.049000,-0.069000,-0.060000,-0.020000
R_MST_ROI,0.110000,0.000000,0.114000,0.049000,0.032000,-0.046000,-0.111000,0.021000
R_V6_ROI,0.033000,0.114000,0.000000,0.042000,0.011000,0.100000,-0.018000,0.026000
R_V2_ROI,0.045000,0.049000,0.042000,0.000000,-0.047000,-0.045000,-0.077000,-0.022000
R_V3_ROI,0.049000,0.032000,0.011000,-0.047000,0.000000,-0.098000,-0.059000,-0.058000


## 4. sMRI

### 4.1 Morfometri: 5 fitur untuk setiap parcel

Satu baris = satu parcel korteks HCP (360 parcel, sama dengan simpul graf).

In [8]:
NAMA = ["luas (mm²)", "volume GM (mm³)", "ketebalan (mm)", "kelengkungan", "kedalaman sulkus"]
nama_parcel = [str(n) for n in F["parcel_hcp2016"]]
M = F["morfo_hcp2016"]
display(pd.DataFrame(M[CONTOH][:10], index=nama_parcel[:10], columns=NAMA).round(3)
        .style.set_caption(f"Morfometri 10 parcel pertama, {kohort.loc[CONTOH, 'FILE_ID']} (bentuk lengkap per subjek {M.shape[1:]})"))
display(pd.DataFrame({"Fitur": NAMA, "Rata-rata per parcel (semua subjek)": M.mean((0, 1)), "SD": M.std((0, 1)),
                      "Minimum": M.min((0, 1)), "Maksimum": M.max((0, 1))}).round(3)
        .style.set_caption(f"Ringkasan morfometri {M.shape[0]} subjek × {M.shape[1]} parcel"))

,luas (mm²),volume GM (mm³),ketebalan (mm),kelengkungan,kedalaman sulkus
R_V1_ROI,3648.295898,5764.689941,1.580000,0.042000,-0.015000
R_MST_ROI,326.859985,768.419006,2.351000,-0.062000,-0.628000
R_V6_ROI,483.125000,889.859009,1.842000,-0.025000,-0.435000
R_V2_ROI,2929.934082,6267.136230,2.139000,0.030000,0.047000
R_V3_ROI,1921.444946,4154.360840,2.162000,0.028000,-0.107000
R_V4_ROI,1433.871948,3455.785889,2.410000,-0.020000,-0.244000
R_V8_ROI,322.532013,789.439026,2.448000,0.038000,-0.063000
R_4_ROI,2466.616943,4110.288086,1.666000,0.026000,0.294000
R_3b_ROI,1570.418945,2309.309082,1.471000,0.041000,0.244000
R_FEF_ROI,641.137024,1169.689941,1.824000,-0.067000,-0.462000


,Fitur,Rata-rata per parcel (semua subjek),SD,Minimum,Maksimum
0,luas (mm²),524.966003,461.921997,7.063000,43698.769531
1,volume GM (mm³),1459.614014,1152.707031,0.000000,89503.578125
2,ketebalan (mm),2.848000,0.515000,0.000000,4.902000
3,kelengkungan,0.024000,0.065000,-0.342000,0.620000
4,kedalaman sulkus,-0.085000,0.437000,-1.675000,1.386000


Sebelum masuk simpul graf (S2–S4), luas dan volume diubah menjadi **proporsi terhadap total otak subjek** (koreksi ukuran kepala), lalu kelima fitur distandardisasi per parcel dengan statistik data latih (notebook 17).

### 4.2 Pemeriksaan kewajaran sMRI

Tahap sMRI DCAN **tidak punya QC otomatis** seperti QC gerakan pada rs-fMRI. Pemeriksaan sederhana di bawah mencari subjek yang permukaan korteksnya tidak wajar:
- **luas korteks total ekstrem**, yaitu jauh dari median semua subjek (|z robust| > 4, memakai median dan MAD);
- **parcel dengan ketebalan ≈ 0** (< 0,5 mm), padahal ketebalan korteks manusia ±1,5–4,5 mm.

Pemeriksaan ini **deskriptif**. Subjek yang ditemukan tetap ada di kohort dan ikut dipakai S2–S4, karena kriteria kohort sudah ditetapkan sebelumnya (notebook 09). Temuan ini dicatat sebagai **keterbatasan**. Pengaruhnya diperkirakan kecil (4 dari 1.638 subjek), dan luas serta volume diubah menjadi proporsi terhadap total otak sebelum masuk model, sehingga perbedaan skala keseluruhan sebagian terkoreksi. Tetapi data subjek dengan ketebalan ≈ 0 jelas rusak.

In [9]:
luas_total = M[:, :, 0].sum(1)
med = np.median(luas_total)
z = (luas_total - med) / (1.4826 * np.median(np.abs(luas_total - med)))
tipis = (M[:, :, 2] < 0.5).sum(1)
curiga = np.where((np.abs(z) > 4) | (tipis > 0))[0]
display(pd.DataFrame({"FILE_ID": kohort["FILE_ID"].values[curiga], "Dataset": kohort["dataset"].values[curiga], "Situs": kohort["situs"].values[curiga],
                      "Label (1 = ASD)": kohort["label"].values[curiga], "Luas korteks total (mm²)": luas_total[curiga].round(0),
                      "z luas (median semua subjek = 0)": z[curiga].round(1), "Parcel dengan ketebalan < 0,5 mm": tipis[curiga]})
        .style.set_caption(f"Subjek dengan permukaan korteks tidak wajar: {len(curiga)} dari {len(kohort)} (median luas total {med:,.0f} mm²)"))

,FILE_ID,Dataset,Situs,Label (1 = ASD),Luas korteks total (mm²),z luas (median semua subjek = 0),"Parcel dengan ketebalan < 0,5 mm"
0,UM_1_0050312,abide1,UM_1,1,1150118.000000,56.500000,0
1,29190,abide2,ABIDEII-NYU_1,1,282439.000000,5.500000,0
2,29740,abide2,ABIDEII-UCLA_1,1,32689.000000,-9.200000,13
3,29755,abide2,ABIDEII-UCLA_1,0,33906.000000,-9.100000,9


## 5. Ringkasan per Skenario

| Skenario | Fenotipik | rs-fMRI | sMRI |
| :---: | :--- | :--- | :--- |
| S1 | – | FC tangent 3 atlas (cabang linear) · baris FC tangent HCP (simpul) · FC korelasi (sisi) | – |
| S2 | – | sama dengan S1 | + morfometri 5 fitur per simpul |
| S3 | + 4 fitur (cabang fenotipik) | sama dengan S1 | sama dengan S2 |
| S4 | sama dengan S3 | sama dengan S1 | sama dengan S2 |

S4 memakai data yang **sama persis** dengan S3; yang berubah hanya hyperparameter, yang dicari dengan *Optuna* (notebook 19).